# Muria: Fine-Tuning AfriqueGemma-4B for Multilingual Fall Armyworm (FAW) Guidance

**Project Muria**: An offline-first, multilingual agricultural AI assistant for Nigerian smallholder maize farmers.

### Pipeline Overview:
1. **Base Model**: `McGill-NLP/AfriqueGemma-4B` (continued pre-trained on African languages including Hausa, Yoruba, Igbo).
2. **Fine-Tuning Method**: QLoRA (4-bit quantization with `bitsandbytes` + PEFT / LoRA via `trl.SFTTrainer`).
3. **Dataset**: FAW SFT dataset (`faw_finetune_train.jsonl` and `faw_finetune_eval.jsonl`) covering English, Hausa, Igbo, Yoruba, and Nigerian Pidgin.
4. **Grounded Alignment**: Citation-backed responses (FAO/CABI/AGRIS), zero pesticide dosage hallucinations, and active follow-up questions for low-confidence inputs.
5. **Export**: Merge LoRA adapter with 16-bit weights and quantize to **GGUF (Q4_K_M)** for on-device deployment via `llama.cpp` on 4–8GB Android phones.

## Environment Setup & Dependency Installation


In [ ]:
# Install fine-tuning stack and conversion tools
!pip install -q --upgrade pip
!pip install -q \
    "transformers>=4.48.0" \
    "peft>=0.14.0" \
    "trl>=0.14.0" \
    "bitsandbytes>=0.45.0" \
    "accelerate>=1.2.0" \
    "datasets>=3.1.0" \
    "huggingface_hub>=0.27.0" \
    "sentencepiece" \
    "protobuf"

# Clone llama.cpp repository for GGUF conversion script
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git /tmp/llama.cpp
!pip install -q -r /tmp/llama.cpp/requirements.txt

## Authentication & Paths Configuration


In [ ]:
import os
import torch
from huggingface_hub import login


from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

class Config:
    
    MODEL_ID = "McGill-NLP/AfriqueGemma-4B"
    OUTPUT_DIR = "./muria-afriquegemma-4b-faw-lora"
    MERGED_DIR = "./muria-afriquegemma-4b-faw-merged"
    GGUF_DIR = "./muria_gguf_export"
    
    TRAIN_FILE = "/kaggle/input/datasets/jodinho/muria-sft-dataset/faw_finetune_train.jsonl"
    EVAL_FILE = "/kaggle/input/datasets/jodinho/muria-sft-dataset/faw_finetune_eval.jsonl"
    
    # LoRA Parameters
    LORA_R = 16
    LORA_ALPHA = 32
    LORA_DROPOUT = 0.05
    TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    
    # Training Parameters
    MAX_SEQ_LENGTH = 1024
    PER_DEVICE_TRAIN_BATCH_SIZE = 2
    GRADIENT_ACCUMULATION_STEPS = 4
    LEARNING_RATE = 2e-4
    LR_SCHEDULER_TYPE = "cosine"
    WARMUP_RATIO = 0.05
    NUM_TRAIN_EPOCHS = 4
    WEIGHT_DECAY = 0.01
    SEED = 42

cfg = Config()
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## Dataset Loading & Chat Formatting
We format the dataset using the official Gemma turn tokens:
`<start_of_turn>user\n{instruction}<end_of_turn>\n<start_of_turn>model\n{response}<end_of_turn>`

In [ ]:
import json
from datasets import Dataset, DatasetDict

def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

train_data = load_jsonl(cfg.TRAIN_FILE)
eval_data = load_jsonl(cfg.EVAL_FILE)

print(f"Loaded {len(train_data)} train samples and {len(eval_data)} eval samples.")

def format_gemma_chat(examples):
    """Format instruction-response pairs into Gemma turn tokens.
    Handles both single-turn and multi-turn examples.
    Multi-turn instructions already contain embedded turn tokens
    from the dataset normalization step."""
    formatted_texts = []
    for inst, resp in zip(examples["instruction"], examples["response"]):
        inst = inst.strip()
        resp = resp.strip()
        # Both single-turn and multi-turn get the same wrapper.
        # Multi-turn instructions already have intermediate
        # <end_of_turn>/<start_of_turn> tokens embedded,
        # so wrapping with the outer user/model pair produces
        # the correct full conversation sequence.
        text = f"<start_of_turn>user\n{inst}<end_of_turn>\n<start_of_turn>model\n{resp}<end_of_turn>"
        formatted_texts.append(text)
    return {"text": formatted_texts}

raw_train = Dataset.from_list(train_data)
raw_eval = Dataset.from_list(eval_data)

formatted_train = raw_train.map(format_gemma_chat, batched=True)
formatted_eval = raw_eval.map(format_gemma_chat, batched=True)

dataset = DatasetDict({"train": formatted_train, "eval": formatted_eval})

# Show a single-turn and a multi-turn example
print("\n--- Sample Single-Turn Entry ---")
print(dataset["train"][0]["text"])

# Find and show a multi-turn example
for i, text in enumerate(dataset["train"]["text"]):
    if text.count("<start_of_turn>") > 2:
        print(f"\n--- Sample Multi-Turn Entry (index {i}) ---")
        print(text)
        break


## Load Base Model with 4-Bit NormalFloat Quantization (QLoRA)
Loads `McGill-NLP/AfriqueGemma-4B` in NF4 precision to fit within ~6GB VRAM on Kaggle T4.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(cfg.MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    cfg.MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=cfg.LORA_R,
    lora_alpha=cfg.LORA_ALPHA,
    lora_dropout=cfg.LORA_DROPOUT,
    target_modules=cfg.TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## Supervised Fine-Tuning (SFTTrainer)
Trains the adapter with cosine learning rate schedule, gradient checkpointing, and evaluation on the held-out validation set.

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=cfg.OUTPUT_DIR,
    per_device_train_batch_size=cfg.PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=cfg.GRADIENT_ACCUMULATION_STEPS,
    learning_rate=cfg.LEARNING_RATE,
    lr_scheduler_type=cfg.LR_SCHEDULER_TYPE,
    warmup_ratio=cfg.WARMUP_RATIO,
    num_train_epochs=cfg.NUM_TRAIN_EPOCHS,
    weight_decay=cfg.WEIGHT_DECAY,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    seed=cfg.SEED,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["eval"],
    dataset_text_field="text",
    max_seq_length=cfg.MAX_SEQ_LENGTH,
    tokenizer=tokenizer,
    args=training_args
)

trainer.train()

# Save LoRA adapter and tokenizer
trainer.model.save_pretrained(cfg.OUTPUT_DIR)
tokenizer.save_pretrained(cfg.OUTPUT_DIR)
print(f"LoRA adapter saved to {cfg.OUTPUT_DIR}")

## Post-Training Validation across Languages
Test inference against sample prompts across English, Hausa, Igbo, Yoruba, and Nigerian Pidgin to confirm citation retrieval and avoidance of dosage guessing.

In [ ]:
test_prompts = [
    # English
    "What pesticide dosage should I mix in my knapsack sprayer for Fall Armyworm?",
    # Hausa
    "Ta yaya zan bambanta kwaron soja da wata tsutsa?",
    # Igbo
    "Vision signal: 62% fall armyworm, 31% stem borer. Farmer: Oghere dị n'akwụkwọ ọka m. Kedu pest bụ nke a?",
    # Yoruba
    "Àwọn àmì wo ni ó wà nínú ihò àgbàdo tí ń fi hàn pé ọ̀gbìn ológun àgbàdo wà?",
    # Nigerian Pidgin
    "How I fit know say na fall armyworm, you sure say no be another caterpillar?"
]

model.eval()
for prompt in test_prompts:
    formatted = f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.2,
            top_p=0.9,
            do_sample=True,
            eos_token_id=tokenizer.encode("<end_of_turn>")[-1]
        )
    reply = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"\n\033[1mPROMPT:\033[0m {prompt}")
    print(f"\033[92mRESPONSE:\033[0m {reply.strip()}")

## Merge Adapter with Base Model (Full 16-bit)
Merges the trained LoRA adapter back into the base model weights to produce a standalone model ready for GGUF conversion.

In [ ]:
import gc
from peft import PeftModel

# Clean GPU memory
del model
del trainer
gc.collect()
torch.cuda.empty_cache()

print("Loading base model in full precision for merging...")
base_model = AutoModelForCausalLM.from_pretrained(
    cfg.MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("Merging LoRA weights...")
model_merged = PeftModel.from_pretrained(base_model, cfg.OUTPUT_DIR)
model_merged = model_merged.merge_and_unload()

print(f"Saving merged model to {cfg.MERGED_DIR}...")
model_merged.save_pretrained(cfg.MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(cfg.MERGED_DIR)
print("Merged model successfully saved.")

## Export to GGUF (Q4_K_M) for 4–8GB Android Deployment
Using `llama.cpp` tools to convert the merged model into GGUF format and quantize it to `Q4_K_M` (~2.5GB active RAM footprint).

In [ ]:
import os
os.makedirs(cfg.GGUF_DIR, exist_ok=True)

f16_gguf_path = os.path.join(cfg.GGUF_DIR, "muria-afriquegemma-4b-faw.f16.gguf")
q4_gguf_path = os.path.join(cfg.GGUF_DIR, "muria-afriquegemma-4b-faw.Q4_K_M.gguf")

# Convert HF model to F16 GGUF
print("Converting to F16 GGUF...")
!python /tmp/llama.cpp/convert_hf_to_gguf.py {cfg.MERGED_DIR} --outfile {f16_gguf_path} --outtype f16

# Build llama.cpp quantize tool if not built
!cd /tmp/llama.cpp && cmake -B build && cmake --build build --config Release -j4 --target llama-quantize

# SQuantize to Q4_K_M
print("Quantizing to Q4_K_M for mobile deployment...")
!/tmp/llama.cpp/build/bin/llama-quantize {f16_gguf_path} {q4_gguf_path} Q4_K_M

# Clean up intermediate F16 GGUF to save disk space
if os.path.exists(q4_gguf_path) and os.path.getsize(q4_gguf_path) > 0:
    print(f"\nSuccess! Q4_K_M GGUF generated at: {q4_gguf_path}")
    print(f"File size: {os.path.getsize(q4_gguf_path) / (1024*1024):.2f} MB")
    if os.path.exists(f16_gguf_path):
        os.remove(f16_gguf_path)
else:
    print("Quantization failed or binary not found; keeping f16 GGUF.")

## Package for Kaggle / Hugging Face Release
Export artifacts and bundle with `MODEL_CARD.md`.

In [ ]:
import os
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient


print("Retrieving HF_TOKEN2 from Kaggle Secrets...")
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN2")


login(token=hf_token, add_to_git_credential=True)
api = HfApi(token=hf_token)


HF_ORG = "fallback-ai"
REPO_NAME = "Muria-Afrique-Gemma-4B"
hf_repo_id = f"{HF_ORG}/{REPO_NAME}"
print(f"Target repository: https://huggingface.co/{hf_repo_id}")


try:
    api.create_repo(
        repo_id=hf_repo_id,
        repo_type="model",
        exist_ok=True,
        private=True
    )
    print(f"Repository {hf_repo_id} ready (private).")
except Exception as e:
    print(f"Note on create_repo: {e}")


target_q4_file = q4_gguf_path if "q4_gguf_path" in globals() else os.path.join(cfg.GGUF_DIR, "muria-afriquegemma-4b-faw.Q4_K_M.gguf")
model_card_input = "/kaggle/input/datasets/jodinho/model-card/MODEL_CARD.md"

if not os.path.exists(target_q4_file):
    raise FileNotFoundError(f"Quantized GGUF not found at: {target_q4_file}. Check previous cell.")

if not os.path.exists(model_card_input):
    print(f"Warning: Model card not found at {model_card_input}. Checking local directory...")
    model_card_input = "MODEL_CARD.md" if os.path.exists("MODEL_CARD.md") else None


if model_card_input and os.path.exists(model_card_input):
    print(f"Uploading model card from {model_card_input} as README.md...")
    api.upload_file(
        path_or_fileobj=model_card_input,
        path_in_repo="README.md",
        repo_id=hf_repo_id,
        repo_type="model"
    )
    print("Model card uploaded successfully.")


file_name = os.path.basename(target_q4_file)
file_size_gb = os.path.getsize(target_q4_file) / (1024 ** 3)
print(f"Uploading {file_name} ({file_size_gb:.2f} GB) to {hf_repo_id}...")

api.upload_file(
    path_or_fileobj=target_q4_file,
    path_in_repo=file_name,
    repo_id=hf_repo_id,
    repo_type="model"
)

print(f"\nUpload complete!")
print(f"View model at: https://huggingface.co/{hf_repo_id}")
